# Phase 8: Multiclass Known-Attack Classification
### Smart Grid Cyber-Physical Intrusion Detection

This notebook evaluates multiclass models to attribute network traffic and power telemetry to specific attack types:
- **`data4cyber_dataset`**: 4 DER attack families (`benign`, `industroyer_modbus`, `mitm_false_data_injection`, `mqtt_price_signal_manipulation`).
- **`CICFlowMeter_Balanced_DNP3_SCADA`**: 11 DNP3 SCADA attack classes.


In [ ]:
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

if os.path.exists('src'):
    PROJECT_ROOT = os.path.abspath('.')
elif os.path.exists('../src'):
    PROJECT_ROOT = os.path.abspath('..')
else:
    PROJECT_ROOT = os.path.abspath('.')
sys.path.insert(0, PROJECT_ROOT)

from src.data_loader import load_data4cyber, get_preprocessor

sns.set_theme(style="whitegrid")


## 1. Load Multiclass DER Attack Campaigns (`data4cyber_dataset`)


In [ ]:
data = load_data4cyber(random_state=42)
X_train, y_train = data['X_train'], data['y_train_family']
X_test, y_test = data['X_test'], data['y_test_family']
class_names = data['class_names_family']
features = data['feature_names']

print("Class Names:", class_names)
print("Test distribution across families:\n", pd.Series(y_test).value_counts())


## 2. Train Multiclass Classifiers


In [ ]:
prep = get_preprocessor(scaling=False)
X_tr = prep.fit_transform(X_train)
X_te = prep.transform(X_test)

models = {
    "Random_Forest": RandomForestClassifier(n_estimators=100, max_depth=15, class_weight='balanced', random_state=42, n_jobs=-1),
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1, eval_metric='mlogloss')
}

for name, clf in models.items():
    print(f"Training {name}...")
    t0 = time.time()
    clf.fit(X_tr, y_train)
    tr_time = time.time() - t0
    
    y_pred = clf.predict(X_te)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    print(f"  -> {name} Macro F1: {macro_f1:.4f} (Trained in {tr_time:.2f}s)")
    print(f"\nClassification Report for {name}:")
    print(classification_report(y_test, y_pred, target_names=class_names, digits=4))


## 3. Multiclass Confusion Matrix Heatmap


In [ ]:
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, models['Random_Forest'].predict(X_te))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=class_names, yticklabels=class_names)
plt.title("Random Forest Multiclass Attack Family Attribution", fontweight='bold')
plt.xlabel("Predicted Attack Family")
plt.ylabel("Actual Attack Family")
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()
